In [ ]:
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from tqdm import tqdm

# -----------------------------
# 1. Model Wrapper (with hooks)
# -----------------------------
class LLMWrapper:
    def __init__(self, model_name="gpt2"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            output_hidden_states=True,
            output_attentions=True
        ).to(self.device)

    def generate(self, prompt, **kwargs):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)

        outputs = self.model.generate(
            **inputs,
            return_dict_in_generate=True,
            output_hidden_states=True,
            output_attentions=True,
            **kwargs
        )

        return outputs

    def decode(self, output):
        return self.tokenizer.decode(output.sequences[0])


# -----------------------------
# 2. Feature Extraction
# -----------------------------
class FeatureExtractor:
    @staticmethod
    def attention_ratio(attentions):
        last = attentions[-1].mean(dim=1)
        seq_len = last.shape[-1]
        split = seq_len // 2

        context = last[:, :, :split].mean().item()
        generated = last[:, :, split:].mean().item()

        return context / (generated + 1e-6)

    @staticmethod
    def fft_features(hidden_states, k=10):
        h = hidden_states[-1].mean(dim=-1).squeeze()
        fft = torch.fft.fft(h)
        mag = torch.abs(fft)
        return mag[:k].cpu().numpy()

    @staticmethod
    def entropy(logits):
        probs = torch.softmax(logits, dim=-1)
        return -(probs * torch.log(probs + 1e-9)).sum(dim=-1).mean().item()


# -----------------------------
# 3. SAE (Simplified Sparse Autoencoder)
# -----------------------------
class SparseAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.encoder = nn.Linear(input_dim, hidden_dim)
        self.decoder = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        z = torch.relu(self.encoder(x))
        recon = self.decoder(z)
        return z, recon


# -----------------------------
# 4. TSV Steering
# -----------------------------
class TSVSteering:
    def __init__(self, vector, strength=1.0):
        self.vector = vector
        self.strength = strength

    def apply(self, hidden_state):
        return hidden_state + self.strength * self.vector


# -----------------------------
# 5. Probe (Hallucination Classifier)
# -----------------------------
class Probe(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.net(x)


# -----------------------------
# 6. CoT + Verifier Loop
# -----------------------------
def cot_prompt(prompt):
    return f"{prompt}\nLet's think step by step."

def verifier_prompt(question, answer):
    return f"Question: {question}\nAnswer: {answer}\nIs this factually correct? Answer yes or no."


# -----------------------------
# 7. Experiment Runner
# -----------------------------
class Experiment:
    def __init__(self, llm, probe):
        self.llm = llm
        self.probe = probe

    def extract_features(self, outputs):
        attn = FeatureExtractor.attention_ratio(outputs.attentions)
        fft = FeatureExtractor.fft_features(outputs.hidden_states)

        logits = outputs.scores[-1]
        ent = FeatureExtractor.entropy(logits)

        return np.concatenate([[attn, ent], fft])

    def run_example(self, question):
        # Step 1: CoT generation
        cot_out = self.llm.generate(cot_prompt(question), max_new_tokens=50)
        cot_text = self.llm.decode(cot_out)

        # Step 2: Feature extraction
        feats = self.extract_features(cot_out)
        feats_t = torch.tensor(feats, dtype=torch.float32).unsqueeze(0)

        # Step 3: Detection
        logits = self.probe(feats_t)
        prob = torch.softmax(logits, dim=-1)[0, 1].item()

        # Step 4: Verifier loop if needed
        if prob > 0.5:
            verify_out = self.llm.generate(
                verifier_prompt(question, cot_text),
                max_new_tokens=10
            )
            verdict = self.llm.decode(verify_out)

            return {
                "question": question,
                "answer": cot_text,
                "halluc_prob": prob,
                "verified": verdict
            }

        return {
            "question": question,
            "answer": cot_text,
            "halluc_prob": prob,
            "verified": "skipped"
        }

    def run_dataset(self, dataset, n=50):
        results = []
        for i in tqdm(range(n)):
            q = dataset[i]["question"]
            results.append(self.run_example(q))
        return results

In [ ]:
dataset = load_dataset("truthful_qa", "generation")["validation"]

llm = LLMWrapper("gpt2")
probe = Probe(dim=12)

exp = Experiment(llm, probe)

results = exp.run_dataset(dataset, n=20)

for r in results[:3]:
    print(r)

In [ ]:
def get_sae_features(hidden_states, sae):
    h = hidden_states[-1].mean(dim=1)
    z, _ = sae(h)
    return z.detach()

In [ ]:
z[hallucination_indices] = 0

In [ ]:
class SubspaceAnalyzer:
    def __init__(self, sae):
        self.sae = sae

    def analyze(self, hidden_states):
        h = hidden_states[-1].mean(dim=1)  # [batch, dim]
        z, _ = self.sae(h)

        return {
            "latent": z.detach(),
            "norm": torch.norm(h, dim=-1).item()
        }

In [ ]:
def find_hallucination_features(latents, labels):
    """
    latents: [N, features]
    labels: 1 = hallucination, 0 = truthful
    """
    corr = []

    for i in range(latents.shape[1]):
        feature_vals = latents[:, i]
        correlation = np.corrcoef(feature_vals, labels)[0, 1]
        corr.append(correlation)

    return np.argsort(corr)[-10:]  # top hallucination features

In [ ]:
def intervene_on_latents(z, indices, strength=1.0):
    z[:, indices] -= strength * z[:, indices]
    return z

In [ ]:
def hook_fn(module, input, output):
    h = output
    z, recon = sae(h)

    z = intervene_on_latents(z, hallucination_indices)
    new_h = sae.decoder(z)

    return new_h